# EviChange — PlanetScope months → Earth Engine (Phase 13b, D12)

Turns the delivered PlanetScope scenes (Google Drive) into **one cloud-masked 3 m mosaic per month**
and puts them in the Earth Engine collection `projects/space-510615/assets/planet_qn`, where
`agents/planet.py` reads them. Runs on Colab because the project PC can't run rasterio (Windows
Smart App Control blocks its DLLs). **No GPU needed:** pick a CPU runtime, which uses almost no
compute units.

Each mosaic has 9 bands: the 8 surface-reflectance bands (harmonised to Sentinel-2, × 10 000) and
`clear` (1 where Planet's UDM2 cloud mask says clear). Pixels that aren't clear are 0 = no data.
Within a month, the clearest scene wins where scenes overlap.

**February 2025 is skipped for now** (refused over quota; re-order in November, then rerun this
notebook: months already in Earth Engine are left alone).

**Licence.** The mosaics are Planet imagery under the Education and Research programme: they stay in
your Drive and your Earth Engine project. Never commit them to either GitHub repo.

## Step 1 — Install, mount Drive, connect Earth Engine

In [ ]:
!pip -q install rasterio earthengine-api
import glob, json, os, re
import numpy as np
import rasterio
from rasterio.merge import merge
import ee
from google.colab import drive

drive.mount('/content/drive')
EE_PROJECT = 'space-510615'
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print('rasterio', rasterio.__version__, '| Earth Engine ready')

## Step 2 — Settings

`PLANET_DIR` is the folder `planet_order.py download --copy-to` filled, one sub-folder per month.
Leave `GCS_BUCKET` empty unless you have a Cloud Storage bucket in the project (see Step 5).

In [ ]:
PLANET_DIR = '/content/drive/MyDrive/EviChange/planet'          # <month>/ folders from planet_order.py
OUT_DIR    = '/content/drive/MyDrive/EviChange/planet_mosaics'  # monthly mosaics are written here
COLLECTION = f'projects/{EE_PROJECT}/assets/planet_qn'
GCS_BUCKET = ''                     # e.g. 'evichange-planet'; empty = upload through the Code Editor (Step 5b)
SKIP_MONTHS = {'2025-02'}           # not delivered yet (over quota); re-order in November

BANDS = ['coastal_blue', 'blue', 'green_i', 'green', 'yellow', 'red', 'rededge', 'nir', 'clear']
os.makedirs(OUT_DIR, exist_ok=True)
months = sorted(m for m in os.listdir(PLANET_DIR) if re.fullmatch(r'\d{4}-\d{2}', m) and m not in SKIP_MONTHS)
print(len(months), 'months:', ', '.join(months))

## Step 3 — Which months are already in Earth Engine

In [ ]:
def asset_id(month):
    y, m = month.split('-')
    return f'{COLLECTION}/m{y}_{m}'

try:
    ee.data.getAsset(COLLECTION)
except ee.EEException:
    ee.data.createAsset({'type': 'IMAGE_COLLECTION'}, COLLECTION)
    print('created', COLLECTION)

def in_ee(month):
    try:
        ee.data.getAsset(asset_id(month)); return True
    except ee.EEException:
        return False

todo = [m for m in months if not in_ee(m)]
print('already in Earth Engine:', [m for m in months if m not in todo])
print('to do:', todo)

## Step 4 — One mosaic per month

Per scene: surface reflectance where UDM2 band 1 (clear) is 1, else 0. Scenes are merged clearest
first. Takes a few minutes per month; the mosaics land in `OUT_DIR`.

In [ ]:
def scenes(month):
    out = []
    for sr in sorted(glob.glob(f'{PLANET_DIR}/{month}/*_AnalyticMS_SR_8b_harmonized_clip.tif')):
        stem = os.path.basename(sr).split('_3B_')[0]
        udm = glob.glob(f'{PLANET_DIR}/{month}/{stem}_3B_udm2_clip.tif')
        meta = glob.glob(f'{PLANET_DIR}/{month}/{stem}_metadata.json')
        if not udm:
            print('  no UDM2 for', stem, '- skipped'); continue
        props = json.load(open(meta[0]))['properties'] if meta else {}
        out.append({'id': stem, 'sr': sr, 'udm': udm[0], 'clear_percent': props.get('clear_percent', 0),
                    'acquired': props.get('acquired', '')})
    return sorted(out, key=lambda s: -s['clear_percent'])

TMP = '/content/planet_tmp'          # Colab's local disk: faster than Drive, and keeps RAM free
os.makedirs(TMP, exist_ok=True)

def masked_scene(s):
    # 9-band copy of one scene on local disk: SR where clear (at least 1, so a dark clear pixel isn't
    # mistaken for no data), 0 elsewhere; band 9 = clear (0/1)
    with rasterio.open(s['sr']) as src, rasterio.open(s['udm']) as udm:
        assert (src.crs, src.transform, src.shape) == (udm.crs, udm.transform, udm.shape), s['id']
        sr, clear = src.read(), (udm.read(1) == 1)
        profile = src.profile
    data = np.where(clear, np.maximum(sr, 1), 0).astype('uint16')
    data = np.concatenate([data, clear[None].astype('uint16')])
    profile.update(count=9, dtype='uint16', nodata=0, compress='deflate', tiled=True)
    path = f"{TMP}/{s['id']}.tif"
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(data)
    return path

def mosaic(month):
    ss = scenes(month)
    assert ss, f'no scenes for {month}'
    crs = {rasterio.open(s['sr']).crs.to_string() for s in ss}
    assert len(crs) == 1, f'{month}: scenes in several CRS {crs}; reproject first'
    paths = [masked_scene(s) for s in ss]
    data, transform = merge(paths, nodata=0, method='first')
    with rasterio.open(paths[0]) as first:
        profile = first.profile
    for p in paths:
        os.remove(p)
    profile.update(driver='GTiff', height=data.shape[1], width=data.shape[2], transform=transform,
                   tiled=True, blockxsize=512, blockysize=512, compress='deflate', predictor=2, BIGTIFF='IF_SAFER')
    path = f'{OUT_DIR}/planet_qn_{month}.tif'
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(data)
        dst.descriptions = tuple(BANDS)
    clear_share = float(data[8].sum()) / data[8].size
    date = sorted(s['acquired'][:10] for s in ss if s['acquired'])[0] or f'{month}-01'
    info = {'month': month, 'date': date, 'item_ids': [s['id'] for s in ss], 'clear_share': round(clear_share, 4)}
    json.dump(info, open(path.replace('.tif', '.json'), 'w'), indent=1)
    print(f'{month}: {len(ss)} scenes, {data.shape[2]} x {data.shape[1]} px, clear {clear_share:.0%} -> {os.path.basename(path)}')
    return path, info

made = {}
for month in todo:
    made[month] = mosaic(month)

## Step 5 — Into Earth Engine

**5a, with a Cloud Storage bucket** (`GCS_BUCKET` set): the mosaics are copied to the bucket and
ingested automatically. Needs billing on the project and the bucket in a region Earth Engine reads.

**5b, without a bucket** (`GCS_BUCKET` empty): upload each file by hand in the Earth Engine Code
Editor (code.earthengine.google.com) → **Assets → NEW → Image upload (GeoTIFF)**:
- *Source files*: the `planet_qn_<month>.tif` from `OUT_DIR` (download it from Drive first)
- *Asset ID*: exactly the one printed below
- *Masking mode*: **No-data value**, value **0**

Then run Step 6, which sets each image's date and properties.

In [ ]:
def properties(info):
    return {'month': info['month'], 'date': info['date'], 'source': 'planetscope_8b_sr_harmonized',
            'scene_count': len(info['item_ids']), 'item_ids': ','.join(info['item_ids']),
            'clear_share': info['clear_share'],
            'attribution': 'Imagery © Planet Labs PBC (PlanetScope, Education and Research programme)'}

if GCS_BUCKET:
    from google.cloud import storage
    from google.colab import auth
    auth.authenticate_user()
    bucket = storage.Client(project=EE_PROJECT).bucket(GCS_BUCKET)
    for month, (path, info) in made.items():
        blob = bucket.blob(f'planet_qn/{os.path.basename(path)}')
        blob.upload_from_filename(path)
        manifest = {
            'name': asset_id(month),
            'tilesets': [{'id': '0', 'sources': [{'uris': [f'gs://{GCS_BUCKET}/{blob.name}']}]}],
            'bands': [{'id': b, 'tilesetId': '0', 'tilesetBandIndex': i, 'missingData': {'values': [0]}} for i, b in enumerate(BANDS)],
            'startTime': f"{info['date']}T00:00:00Z",
            'properties': properties(info),
        }
        task = ee.data.startIngestion(ee.data.newTaskId()[0], manifest)
        print(month, 'ingestion started:', task['id'])
    print('Ingestion takes ~5-15 min per month: check the Tasks tab in the Code Editor, then run Step 6.')
else:
    print('Upload these in the Code Editor (Assets -> NEW -> Image upload), masking: no-data value 0:\n')
    for month, (path, info) in made.items():
        print(f'  {os.path.basename(path):28s} ->  {asset_id(month)}')

## Step 6 — Set dates and properties, then check the collection

In [ ]:
def finalize(month):
    info = json.load(open(f'{OUT_DIR}/planet_qn_{month}.json'))
    ee.data.updateAsset(asset_id(month), {'startTime': f"{info['date']}T00:00:00Z", 'properties': properties(info)},
                        ['start_time', 'properties'])

for month in months:
    if in_ee(month):
        finalize(month)

col = ee.ImageCollection(COLLECTION)
have = col.aggregate_array('month').getInfo()
print(f'{col.size().getInfo()} images in {COLLECTION}')
print('months:', ', '.join(sorted(have)))
print('missing:', [m for m in months if m not in have] or 'none', '| skipped on purpose:', sorted(SKIP_MONTHS))
first = ee.Image(col.first())
print('bands:', first.bandNames().getInfo(), '| scale (m):', round(first.select(0).projection().nominalScale().getInfo(), 2))

## Afterwards

On the project PC, `agents/planet.py` reads the collection: `available_months()`, `month_image()`,
`indices()` (NDVI, NDWI), `water_image()` and `region_chips()`. The collection is private to the
`space-510615` project, which is right for Planet's licence. A teammate using another Earth Engine
project needs read access shared with them (asset → Share).